# MAGs taxonomic classification

Now that each MAG should correspond to an individual taxa, we will perform a taxonomic classification to see which species are present in our samples.

**All the following codes were run on Euler. So they cannot be run on Jupyterhub.**

## 1. Import Kraken2 Database

We start by importing the PlusPF database from Kraken2 as it contains all the domains we're interested in. The database is capped at 16GB, since the complete one requires too much computational capacity.

In [ ]:
mosh annotate build-kraken-db \
    --p-collection pluspf16 \
    --o-kraken2-db $data_dir/kraken2_db16 \
    --o-bracken-db $data_dir/bracken_db16 \

## 2. Classification using kraken2

We run Kraken2 classification so that it can compare the analyzed genomes to a reference.


In [ ]:
mosh annotate classify-kraken2 \
    --i-seqs $data_dir/mags_derep_all_domains.qza \
    --i-db $data_dir/kraken2_db16.qza \
    --p-threads 30 \
    --p-memory-mapping False \
    --o-reports $data_dir/kraken2-reports-mags_db16.qza \
    --o-outputs $data_dir/kraken2-hits-mags_db16.qza

### 2.1 Removing Problematic MAG

We have now two new artifacts: `FeatureData[Kraken2Report % Properties('mags')]` and `FeatureData[Kraken2Output % Properties('mags')]` that we could not directly run into a Qiime-like taxonomy file. There was one problematic MAG found (that was not classified at a kingdom level). So we removed it; the `remove-mags_1`file contains the feature ID of the problematic MAG.

In [ ]:
mosh annotate filter-kraken2-results \
  --i-reports $data_dir/kraken2-reports-mags_db16.qza \
  --i-outputs $data_dir/kraken2-hits-mags_db16.qza \
  --m-metadata-file $data_dir/remove-mags_1.tsv \
  --p-exclude-ids \
  --o-filtered-reports $data_dir/kraken2-reports-mags_db16_filtered-1.qza \
  --o-filtered-outputs $data_dir/kraken2-hits-mags_db16_filtered-1.qza \

## 3. More Qimme2-like taxonomy

In [ ]:
mosh annotate kraken2-to-mag-features \
    --i-reports $data_dir/kraken2-reports-mags_db16_filtered-1.qza \
    --i-outputs $data_dir/kraken2-hits-mags_db16_filtered-1.qza \
    --o-taxonomy $data_dir/mags-taxonomy-db16-filtered.qza\
    --verbose

### 3.1 Get Visualization

In [ ]:
qiime metadata tabulate \
  --m-input-file $data_dir/mags-taxonomy-db16-filtered.qza \
  --o-visualization $data_dir/mags-taxonomy-db16-filtered.qzv

## 4. Taxa Bar plot (original)

This is the bar plot using the taxonomic classification obtained with the previous steps. 

In [ ]:
qiime taxa barplot \
    --i-table $data_dir/mags_derep_ft_merged_filtered.qza \
    --i-taxonomy $data_dir/mags-taxonomy-db16-filtered.qza \
    --m-metadata-file $data_dir/updog_metadata.tsv \
    --o-visualization $data_dir/mags-taxa-bar-plot-original.qzv

As there were a lot of unclassified MAGs, Milo provided another classification method detailed in step 5.

## 5. Other Taxonomic Classification (by Milo)

The previous command generated a bar plot with mostly unclassified MAGs. So Milo decided to write an alternative R script, assigning taxonomy based on majority of contigs instead of assigning on the least common incestor.

In [ ]:
rm(list = ls())
library(tidyverse)

#---- Parameters ----#
input_dir <- "updog_2025/kraken2-reports-mags_db16/"   # folder with *.report files
pct_threshold <- 50              # % threshold to assign taxon
min_contigs <- 5                 # minimum contigs to consider

rank_order <- c("R","R1","R2","K","K1","K2","P","P1","P2","C","C1","C2","O","O1","O2","F","F1","F2","G","G1","G2","S","S1","S2")
rank_levels <- setNames(seq_along(rank_order), rank_order)

df_test <- read_tsv("updog_2025/kraken2-reports-mags_db16/0024a950-f4ad-4c18-a733-2050ee75184d.report.txt",
               col_names = c("pct", "n_assigned", "n_direct", "rank_code", "taxid", "name"),
               comment = "#",
               trim_ws = TRUE,
               col_types = "dddcic") %>%
  mutate(name = str_trim(name))

#---- Function to parse and assign taxonomy ----#
assign_taxonomy <- function(report_file, pct_threshold = 50, min_contigs = 5) {
  
  df <- read_tsv(report_file,
                 col_names = c("pct", "n_assigned", "n_direct", "rank_code", "taxid", "name"),
                 comment = "#",
                 trim_ws = TRUE,
                 col_types = "dddcic") %>%
    mutate(name = str_trim(name))
  
  # remove root/unclassified rows
  #df <- df %>% filter(!str_detect(rank_code, "R|U"))
  
  # pick best match: lowest rank meeting thresholds
  best <- df %>%
    filter(n_assigned >= min_contigs, pct >= pct_threshold) %>%
    mutate(rank_level = rank_levels[rank_code]) %>%
    arrange(desc(rank_level), desc(pct)) %>%
    slice_head(n = 1)
  
  # If none pass thresholds, pick the best available entry
  if (nrow(best) == 0) {
    best <- df %>%
      mutate(rank_level = rank_levels[rank_code]) %>%
      arrange(desc(rank_level), desc(pct)) %>%
      slice_head(n = 1)
  }
  
  # ---- SAFER lineage extraction ---- #
  # Find row of best$taxid in df
  best_row <- match(best$taxid, df$taxid)
  
  # If the taxid is not present, fallback to using the LAST row (deepest rank)
  if (is.na(best_row)) {
    warning(paste0("Taxid ", best$taxid, " not found in df for file: ", report_file,
                   " — using full df as lineage."))
    best_row <- nrow(df)
  }
  
  lineage <- df %>%
    slice(seq_len(best_row)) %>%
    dplyr::filter(rank_code %in% c("U", "R", "R2", "K", "P", "C", "O", "F", "G", "S")) %>%
    mutate(
      rank_prefix = paste0(tolower(rank_code), "__"),
      clean_name = str_replace_all(name, "\\s+", "_"),
      qiime_piece = paste0(rank_prefix, clean_name)
    )
  
  qiime_path <- paste(lineage$qiime_piece, collapse = "; ")
  
  tibble(
    mag_id = tools::file_path_sans_ext(basename(report_file)),
    assigned_name = best$name,
    rank_code = best$rank_code,
    taxid = best$taxid,
    pct = best$pct,
    contigs = best$n_assigned,
    qiime_taxonomy = qiime_path
  )
}


#---- Apply to all reports ----#
results <- list.files(input_dir, pattern = "\\.report.txt$", full.names = TRUE) %>%
  map_dfr(assign_taxonomy, pct_threshold = pct_threshold, min_contigs = min_contigs) %>%
  dplyr::mutate(mag_id = stringr::str_remove(mag_id, "\\.report")) %>%
  dplyr::mutate(qiime_taxonomy = qiime_taxonomy %>%
                  stringr::str_replace("r2__", "d__") %>%
                  stringr::str_remove_all(" ") %>%
                  stringr::str_remove("u__unclassified;") %>%
                  stringr::str_remove("r__root;") %>%
                  stringr::str_replace("u__unclassified", "Unassigned") %>%
                  stringr::str_replace("r__root", "Unassigned"))
tax_artifact <- results %>%
  dplyr::select(mag_id, qiime_taxonomy) %>%
  dplyr::rename(`Feature ID` = mag_id, Taxon = qiime_taxonomy)
readr::write_tsv(tax_artifact, "updog_2025/mags_taxonomy.tsv")

We then renamed his file `kraken2-mags-taxonomy_db16.qza`to `kraken2-mags-taxonomy_db16_milo.qza`. 

### 5.1 Remove unclassified MAGs
We remove the unassigned MAGs, in order to have only the information on matched organisms. 

In [ ]:
qiime taxa filter-table \
    --i-table $data_dir/mags_derep_ft_merged_filtered.qza \
    --i-taxonomy $data_dir/kraken2-mags-taxonomy_db16_milo.qza \
    --p-exclude Unassigned \
    --o-filtered-table $data_dir/mags-table-filtered-milo-no-unassigned.qza

### 5.2 Taxa Bar Plot

In [ ]:
qiime taxa barplot \
    --i-table $data_dir/mags-table-filtered-milo-no-unassigned.qza \
    --i-taxonomy $data_dir/kraken2-mags-taxonomy_db16_milo.qza \
    --m-metadata-file $data_dir/updog_metadata.tsv \
    --o-visualization $data_dir/mags-taxa-bar-plot-milo-no-unclassified.qzv